# Stable Diffusion Dual-Pipeline Gradio App
### Text-to-Image (GPU 0) + Depth-to-Image Restyling (GPU 1)

## Installing Required Libraries

In [2]:
!pip install -q gradio diffusers transformers accelerate torch torchvision \
               xformers safetensors Pillow opencv-python-headless

print("All packages installed successfully!")

All packages installed successfully!


## Verifying GPU Availability

In [3]:
import torch

print("=" * 50)
print(f"PyTorch version    : {torch.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")
print(f"Number of GPUs     : {torch.cuda.device_count()}")
print("=" * 50)

if torch.cuda.device_count() >= 2:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} — {props.total_memory // (1024**3)} GB VRAM")
    print("\nBoth GPUs detected. Ready to proceed!")
elif torch.cuda.device_count() == 1:
    print("Only 1 GPU detected.")
else:
    print("No GPU detected!")

PyTorch version    : 2.10.0+cu128
CUDA available     : True
Number of GPUs     : 2
  GPU 0: Tesla T4 — 14 GB VRAM
  GPU 1: Tesla T4 — 14 GB VRAM

Both GPUs detected. Ready to proceed!


## Importing All Libraries

In [25]:
import torch
import requests
import gradio as gr
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline, StableDiffusionDepth2ImgPipeline
import gc
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"   Gradio version     : {gr.__version__}")
print(f"   Diffusers version  : ", end="")
import diffusers; print(diffusers.__version__)

All libraries imported successfully!
   Gradio version     : 5.50.0
   Diffusers version  : 0.36.0


## Loading Both Pipelines

In [26]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")

login(token=secret_value_0)  

In [29]:
TEXT2IMG_MODEL_ID   = "runwayml/stable-diffusion-v1-5"
DEPTH2IMG_MODEL_ID  = "sd2-community/stable-diffusion-2-depth"

# ─────────────────────────────────────────────────────────────
#  LOAD Pipeline 1 — Text-to-Image on GPU 0
# ─────────────────────────────────────────────────────────────
print("Loading Text-to-Image pipeline on GPU 0 ...")
print(f"   Model: {TEXT2IMG_MODEL_ID}")

pipe_txt2img = StableDiffusionPipeline.from_pretrained(
    TEXT2IMG_MODEL_ID,
    torch_dtype=torch.float16,       
    safety_checker=None,             
    requires_safety_checker=False
)

with torch.cuda.device(0):
    pipe_txt2img = pipe_txt2img.to("cuda:0")
    pipe_txt2img.enable_attention_slicing()   

print(f"   Text-to-Image loaded on GPU 0")
print(f"   VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB\n")


# ─────────────────────────────────────────────────────────────
#  LOAD Pipeline 2 — Depth-to-Image on GPU 1
# ─────────────────────────────────────────────────────────────
print("Loading Depth-to-Image pipeline on GPU 1 ...")
print(f"   Model: {DEPTH2IMG_MODEL_ID}")

pipe_depth2img = StableDiffusionDepth2ImgPipeline.from_pretrained(
    DEPTH2IMG_MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)

with torch.cuda.device(1):
    pipe_depth2img = pipe_depth2img.to("cuda:1")
    pipe_depth2img.enable_attention_slicing()

print(f"   Depth-to-Image loaded on GPU 1")
print(f"   VRAM used: {torch.cuda.memory_allocated(1) / 1e9:.2f} GB\n")

print("=" * 50)
print(" Both pipelines ready!")
print(f"   GPU 0 total VRAM used : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"   GPU 1 total VRAM used : {torch.cuda.memory_allocated(1) / 1e9:.2f} GB")

Loading Text-to-Image pipeline on GPU 0 ...
   Model: runwayml/stable-diffusion-v1-5


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Text-to-Image loaded on GPU 0
   VRAM used: 2.16 GB

Loading Depth-to-Image pipeline on GPU 1 ...
   Model: sd2-community/stable-diffusion-2-depth


...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-depth/snapshots/6cb92dd9430a7f6da8d9e99d7b60acdebcc348b7/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/414 [00:00<?, ?it/s]

   Depth-to-Image loaded on GPU 1
   VRAM used: 2.85 GB

 Both pipelines ready!
   GPU 0 total VRAM used : 2.16 GB
   GPU 1 total VRAM used : 2.85 GB


## Defining Inference Functions

In [30]:
def free_gpu_memory(device_id: int):
    """Release unused VRAM on the specified GPU after inference."""
    with torch.cuda.device(device_id):
        torch.cuda.empty_cache()
        gc.collect()


# ─────────────────────────────────────────────────────────────
#  FUNCTION 1 — Text → Image (GPU 0)
# ─────────────────────────────────────────────────────────────
def generate_text_to_image(
    prompt: str,
    negative_prompt: str,
    num_steps: int,
    guidance_scale: float,
    seed: int
) -> Image.Image:
    """
    Generate an image from a text prompt using GPU 0.
    
    Args:
        prompt          : Text description of the desired image
        negative_prompt : What to avoid in the image
        num_steps       : Number of denoising steps (more = better quality, slower)
        guidance_scale  : How strictly to follow the prompt (7–12 recommended)
        seed            : Random seed for reproducibility (-1 = random)
    """
    if not prompt.strip():
        raise gr.Error("Please enter a text prompt!")

    # Set seed
    generator = None
    if seed != -1:
        generator = torch.Generator(device="cuda:0").manual_seed(int(seed))

    with torch.cuda.device(0):
        result = pipe_txt2img(
            prompt=prompt,
            negative_prompt=negative_prompt if negative_prompt.strip() else None,
            num_inference_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            generator=generator,
            width=512,
            height=512
        )
        image = result.images[0]

    free_gpu_memory(0)
    return image


# ─────────────────────────────────────────────────────────────
#  FUNCTION 2 — Image → Depth-based Restyle (GPU 1)
# ─────────────────────────────────────────────────────────────
def generate_depth_restyle(
    input_image,
    prompt: str,
    negative_prompt: str,
    strength: float,
    num_steps: int,
    guidance_scale: float,
    seed: int
) -> Image.Image:
    """
    Restyle an uploaded image using depth-aware transformation on GPU 1.
    
    Args:
        input_image     : The uploaded image (PIL or numpy array from Gradio)
        prompt          : Style / description to apply
        negative_prompt : What to avoid
        strength        : How much to change the image (0.0=no change, 1.0=full restyle)
        num_steps       : Denoising steps
        guidance_scale  : Prompt adherence strength
        seed            : Random seed (-1 = random)
    """
    if input_image is None:
        raise gr.Error("Please upload an image first!")
    if not prompt.strip():
        raise gr.Error("Please enter a restyle prompt!")

    # Convert numpy array from Gradio to PIL
    if isinstance(input_image, np.ndarray):
        input_image = Image.fromarray(input_image)

    # Resize to model's expected size
    input_image = input_image.convert("RGB").resize((512, 512))

    # Set seed
    generator = None
    if seed != -1:
        generator = torch.Generator(device="cuda:1").manual_seed(int(seed))

    with torch.cuda.device(1):
        result = pipe_depth2img(
            prompt=prompt,
            image=input_image,
            negative_prompt=negative_prompt if negative_prompt.strip() else None,
            strength=float(strength),
            num_inference_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            generator=generator
        )
        image = result.images[0]

    free_gpu_memory(1)
    return image


print("Inference functions defined!")

Inference functions defined!


## Building & Launching the Gradio Interface

In [35]:
custom_css = """
body, .gradio-container {
    font-family: 'Segoe UI', system-ui, sans-serif;
    background: #0f0f0f;
}
.gradio-container {
    max-width: 900px !important;
    margin: 0 auto;
}
h1 { 
    color: #e2e8f0 !important;
    text-align: center;
    font-size: 2rem !important;
    font-weight: 700 !important;
    margin-bottom: 0.25rem !important;
}
.subtitle {
    text-align: center;
    color: #94a3b8;
    font-size: 0.95rem;
    margin-bottom: 1.5rem;
}
.tab-nav button {
    font-weight: 600 !important;
    font-size: 0.95rem !important;
}
.generate-btn {
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    border: none !important;
    color: white !important;
    font-weight: 600 !important;
    font-size: 1rem !important;
    padding: 0.65rem 1.5rem !important;
    border-radius: 8px !important;
    cursor: pointer !important;
    transition: opacity 0.2s !important;
}
.generate-btn:hover { opacity: 0.88 !important; }
.gpu-badge {
    display: inline-block;
    background: #1e293b;
    color: #38bdf8;
    font-size: 0.78rem;
    font-weight: 600;
    padding: 2px 10px;
    border-radius: 999px;
    border: 1px solid #334155;
    margin-bottom: 0.75rem;
}
"""


# ─────────────────────────────────────────────────────────────
#  BUILD INTERFACE
# ─────────────────────────────────────────────────────────────
with gr.Blocks(css=custom_css, title="Stable Diffusion Dual Pipeline") as demo:

    # ── Header ───────────────────────────────────────────────
    gr.HTML("""
        <h1>Stable Diffusion Dual Pipeline</h1>
        <p class='subtitle'>
            Text → Image on <b>GPU 0</b> &nbsp;|&nbsp; Depth Restyle on <b>GPU 1</b>
        </p>
    """)

    # ── Tabs ─────────────────────────────────────────────────
    with gr.Tabs():

        # ════════════════════════════════════════
        #  TAB 1 — Text → Image
        # ════════════════════════════════════════
        with gr.Tab("Text → Image"):

            gr.HTML("<div class='gpu-badge'> Running on GPU 0 — StableDiffusionPipeline</div>")

            with gr.Row():

                # Left column — inputs
                with gr.Column(scale=1):
                    t2i_prompt = gr.Textbox(
                        label="Prompt",
                        placeholder="a cinematic photo of an astronaut riding a horse on Mars, golden hour, 8k",
                        lines=4
                    )
                    t2i_neg_prompt = gr.Textbox(
                        label="Negative Prompt  (optional)",
                        placeholder="blurry, low quality, watermark, distorted",
                        lines=2
                    )
                    with gr.Row():
                        t2i_steps = gr.Slider(
                            label="Inference Steps",
                            minimum=10, maximum=100, value=30, step=1
                        )
                        t2i_guidance = gr.Slider(
                            label="Guidance Scale",
                            minimum=1.0, maximum=20.0, value=7.5, step=0.5
                        )
                    t2i_seed = gr.Number(
                        label="Seed  (-1 = random)",
                        value=-1, precision=0
                    )
                    t2i_btn = gr.Button(
                        "Generate Image",
                        elem_classes="generate-btn",
                        variant="primary"
                    )

                # Right column — output
                with gr.Column(scale=1):
                    t2i_output = gr.Image(
                        label="Generated Image",
                        type="pil",
                        height=512
                    )

            # Example prompts
            gr.Examples(
                examples=[
                    ["a majestic lion in a neon-lit cyberpunk city, rain, photorealistic, 8k", "blurry, low quality", 30, 7.5, 42],
                    ["impressionist oil painting of sunflowers in Provence, van Gogh style", "modern, photo", 40, 8.0, 123],
                    ["underwater city with glowing bioluminescent sea creatures, cinematic", "dark, ugly", 35, 7.5, -1],
                ],
                inputs=[t2i_prompt, t2i_neg_prompt, t2i_steps, t2i_guidance, t2i_seed],
                label="Example Prompts (click to load)"
            )

            # Wire button
            t2i_btn.click(
                fn=generate_text_to_image,
                inputs=[t2i_prompt, t2i_neg_prompt, t2i_steps, t2i_guidance, t2i_seed],
                outputs=t2i_output
            )


        # ════════════════════════════════════════
        #  TAB 2 — Image → Depth-based Restyle
        # ════════════════════════════════════════
        with gr.Tab("Image - Depth Restyle"):

            gr.HTML("<div class='gpu-badge'> Running on GPU 1 — StableDiffusionDepth2ImgPipeline</div>")

            with gr.Row():

                # Left column — inputs
                with gr.Column(scale=1):
                    d2i_image = gr.Image(
                        label="Upload Source Image",
                        type="numpy",
                        height=300
                    )
                    d2i_prompt = gr.Textbox(
                        label="Restyle Prompt",
                        placeholder="a photo of a robot, futuristic, detailed, 8k",
                        lines=3
                    )
                    d2i_neg_prompt = gr.Textbox(
                        label="Negative Prompt  (optional)",
                        placeholder="low quality, blurry, watermark",
                        lines=2
                    )
                    d2i_strength = gr.Slider(
                        label="Restyle Strength  (0 = no change, 1 = full restyle)",
                        minimum=0.1, maximum=1.0, value=0.7, step=0.05
                    )
                    with gr.Row():
                        d2i_steps = gr.Slider(
                            label="Inference Steps",
                            minimum=10, maximum=100, value=30, step=1
                        )
                        d2i_guidance = gr.Slider(
                            label="Guidance Scale",
                            minimum=1.0, maximum=20.0, value=7.5, step=0.5
                        )
                    d2i_seed = gr.Number(
                        label="Seed  (-1 = random)",
                        value=-1, precision=0
                    )
                    d2i_btn = gr.Button(
                        "Restyle Image",
                        elem_classes="generate-btn",
                        variant="primary"
                    )

                # Right column — output
                with gr.Column(scale=1):
                    d2i_output = gr.Image(
                        label="Restyled Image",
                        type="pil",
                        height=512
                    )

            # Wire button
            d2i_btn.click(
                fn=generate_depth_restyle,
                inputs=[d2i_image, d2i_prompt, d2i_neg_prompt, d2i_strength, d2i_steps, d2i_guidance, d2i_seed],
                outputs=d2i_output
            )


    # ── Footer ───────────────────────────────────────────────
    gr.HTML("""
        <div style='text-align:center; margin-top:1.5rem; color:#64748b; font-size:0.8rem;'>
            GPU 0: runwayml/stable-diffusion-v1-5 &nbsp;|&nbsp;
            GPU 1: stabilityai/stable-diffusion-2-depth
        </div>
    """)


# ─────────────────────────────────────────────────────────────
#  LAUNCH — share=True creates a public URL
# ─────────────────────────────────────────────────────────────
print("Launching Gradio app...")
print("The URL will works upto 72 hours")

demo.launch(
    share=True,          
    debug=True,          
    show_error=True      
)

Launching Gradio app...
The URL will works upto 72 hours
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://d04f31f9f47b3d2349.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://d04f31f9f47b3d2349.gradio.live


## Cleanup 

In [36]:
# Gracefully shut down and free memory
demo.close()

del pipe_txt2img
del pipe_depth2img

for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()

gc.collect()
print("Cleanup complete. All GPU memory released.")

Closing server running on port: 7861
Cleanup complete. All GPU memory released.
